This notebook demonstrates how to run the forest deforestation User Defined Process

In [ ]:
import logging

from utils import urls

import openeo

logging.basicConfig(level=logging.INFO)

In [ ]:
connection = openeo.connect("openeo.dataspace.copernicus.eu")

In [ ]:
connection.authenticate_oidc()

# Parameters

In [ ]:
spatial_extent = {
    "west": 30.55,
    "south": 1.07,
    "east": 31.23,
    "north": 1.55,
}

In [ ]:
# minimum canopy cover to be considered forest
# should be one of (10, 20, 30, 40, 50, 60, 70, 80, 90)
# a value of 30 means > 30% canopy cover
canopy_cover_threshold = 30

# minimum likelihood to be considered natural forest
natural_forest_threshold = 0.08

# minimum connected area to be considered forest (m^2)
min_connected_area = 10000

In [ ]:
spatial_resolution = 30  # m

In [ ]:
temporal_variability_threshold = 0.5
flattening_threshold = 0.12
logistic_sse_threshold = 18.3

In [ ]:
# more restrictive (excludes more pixels from being detected as deforestation) than default
temporal_variability_threshold = 0.6
flattening_threshold = 0.14

In [ ]:
cropland_probability_threshold = 0.1

# Script

In [ ]:
kpis_vector_cube = connection.datacube_from_process(
    "End-to-end forest-loss KPIs",
    namespace=urls.END_TO_END_UDP,
    spatial_extent=spatial_extent,
    canopy_cover_threshold=canopy_cover_threshold,
    natural_forest_threshold=natural_forest_threshold,
    min_connected_area=min_connected_area,
    spatial_resolution=spatial_resolution,
    temporal_variability_threshold=temporal_variability_threshold,
    flattening_threshold=flattening_threshold,
    logistic_sse_threshold=logistic_sse_threshold,
    cropland_probability_threshold=cropland_probability_threshold,
)

# Batch job

In [ ]:
job = kpis_vector_cube.create_job(out_format="Parquet")

In [ ]:
job.start_and_wait()

In [ ]:
results = job.get_results()

In [ ]:
!mkdir -p output-udp/
!rm -r output-udp/

In [ ]:
results.download_files("output-udp/")

In [ ]:
import json

with open("logs.json", "w") as f:
    json.dump(job.logs(), f, indent=2)